In [8]:
import sys
sys.path.append("/content/drive/MyDrive/algo_trading_project/strategies")


In [9]:
!dir ..\strategies


 Volume in drive C is Windows
 Volume Serial Number is 96E0-2891

 Directory of c:\Users\yugde\Desktop\Nifty50-Algo-Trading-Strategy\strategies

28-09-2026  15:10    <DIR>          .
28-09-2026  21:42    <DIR>          ..
28-09-2026  15:10             3,016 Strategy_functions.ipynb
28-09-2026  15:10               642 strategy_functions.py
28-09-2026  18:02    <DIR>          __pycache__
               2 File(s)          3,658 bytes
               3 Dir(s)   7,875,805,184 bytes free


In [10]:
import sys
import os

# Add the strategies folder to Python's path
sys.path.append(os.path.abspath("../strategies"))

from strategy_functions import generate_macd_rsi_signals, run_backtest


In [ ]:
import numpy as np
import pandas as pd
# Load features
df = pd.read_csv("../data/nifty50_features.csv", parse_dates=["Date"])
tickers = df["Ticker"].unique()

results = []
for ticker in tickers:
    df_t = df[df["Ticker"] == ticker].sort_values("Date")
    df_t = generate_macd_rsi_signals(df_t)
    df_bt = run_backtest(df_t)

    final_val = df_bt["Portfolio_Value"].iloc[-1]
    total_return = final_val - 1
    n_trades = int((df_bt["Position"].diff().abs() > 0).sum() + (df_bt["Position"].iloc[0] != 0))
    bh_return = df_t["Close"].iloc[-1] / df_t["Close"].iloc[0] - 1
    sharpe = (
        df_bt["Strategy_Return"].mean() / df_bt["Strategy_Return"].std() * (252 ** 0.5)
        if df_bt["Strategy_Return"].std() != 0 else 0
    )
    max_drawdown = (df_bt["Portfolio_Value"] / df_bt["Portfolio_Value"].cummax() - 1).min()

    results.append({
        "Ticker": ticker,
        "Final Value": final_val,
        "Total Return": total_return,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown,
        "Trades": n_trades,
        "Buy&Hold Return": bh_return
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="Sharpe Ratio", ascending=False, inplace=True)
results_df.to_csv("../results/strategy_results.csv", index=False)
results_df


,Ticker,Final Value,Total Return,Sharpe Ratio,Max Drawdown,Trades,Buy&Hold Return
2,BHARTIARTL.NS,2.840949,1.840949,0.734163,-0.333453,0,1.840949
14,SBIN.NS,2.777390,1.777390,0.698511,-0.594925,1,1.498617
11,MARUTI.NS,1.000000,0.000000,0.000000,0.000000,0,0.246247
0,AXISBANK.NS,1.000000,0.000000,0.000000,0.000000,0,1.046947
16,TCS.NS,0.876421,-0.123579,-0.111546,-0.287399,1,1.878212
4,HDFCBANK.NS,0.926504,-0.073496,-0.153160,-0.146729,1,0.898293
17,TITAN.NS,0.638969,-0.361031,-0.229352,-0.687201,2,3.672970
5,HINDUNILVR.NS,0.594784,-0.405216,-0.274958,-0.593850,2,1.176187
9,KOTAKBANK.NS,0.439557,-0.560443,-0.363245,-0.661759,1,0.811945
19,WIPRO.NS,0.412443,-0.587557,-0.440582,-0.817802,2,1.190225
